In [50]:
import pltkit
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib as mpl
import xarray as xr
from scipy.stats import qmc
import sys, os, glob, re, prism, cartopy, pyreadr
from adjustText import adjust_text
sys.path.append(os.path.abspath(".."))
import models.Carleton2022.model.mortality_functions as mf
import models.Burkart2022.model.PAF_calculations as bpaf

In [42]:
"""
PARAMS
"""
wdir = 

### Burkart 2021

In [ ]:
"""
TEMPERATURE ZONES
"""

# Open precalculated temperature zones netcdf file
era5_tz = xr.open_dataset(wdir + "models/burkart2022/data/TemperatureZones/ERA5_mean_1980-2019_land_t2m_tz.nc")

fig = mpl.pyplot.figure(figsize=(12,6))
ax = fig.add_subplot(111, projection=cartopy.crs.Robinson(central_longitude=0), frameon=True)
ax.set_rasterization_zorder(3)
ax.coastlines(resolution='50m', lw = 0.1)
ax.add_feature(cartopy.feature.OCEAN, facecolor='white', zorder=2)
ax.spines['geo'].set_linewidth(0.2)

# Plot the temperature zones
im = era5_tz.t2m.plot(
    ax=ax,
    transform=cartopy.crs.PlateCarree(),
    infer_intervals=False,
    cbar_kwargs = {
        "orientation": "horizontal", 
        "shrink":0.5, 
        "pad":0.06, 
        "aspect":20, 
        "label":"Temperature zones", 
        "ticks": [6, 8, 10, 12, 14, 16, 18, 20, 22, 24, 26, 28]
        },
    vmin=6,
    vmax=28, 
    levels=23,
    extend="both",
    cmap = "cividis"
    )

# Change colour Antarctica 
land_shp = cartopy.io.shapereader.natural_earth(resolution='110m', category='physical', name='land')
land_geoms = cartopy.io.shapereader.Reader(land_shp).geometries()
for land in land_geoms:
    if land.bounds[1] < -60:
        ax.add_geometries([land], cartopy.crs.PlateCarree(), 
                          facecolor='whitesmoke', 
                          edgecolor='none', 
                          zorder=2)

# Save
mpl.rcParams['pdf.compression'] = True
mpl.pyplot.savefig(wdir +"/figures/Paper1/SM_Burkart_TemperatureZones.pdf", dpi=300, bbox_inches='tight')
mpl.pyplot.show()

In [ ]:
"""
BURKART ERFs
"""

sets = bpaf.ModelSettings(
        wdir=wdir+"models/burkart2022/",
        temp_dir="X:/user/liprandicn/Data/ERA5/t2m_daily",
        project=None,
        scenario="SSP2_ERA5",
        years=[2010],
        draw="mean",
        single_erf=False, 
        extrap_erf=False,
    )

# Load ERFs
erf = bpaf.LoadExposureResponseFunctionsAll(sets)

for cause in pltkit.causes.keys():
    erf[cause] = erf[cause].set_index("temperature_zone")
    erf[cause]["mean"] = erf[cause].iloc[:,1:].mean(axis=1)

mpl.pyplot.rcParams["axes.prop_cycle"] = mpl.pyplot.cycler("color", mpl.pyplot.cm.cividis(np.linspace(0,1,23)))

fig, axs = mpl.pyplot.subplots(5,4, figsize=(12,14))
axs = axs.flatten()

for i, cause in enumerate(pltkit.causes.keys()):
    
    for tz in range(6,29):
        
        x = erf[cause]["daily_temperature"].loc[tz].values   # Daily Temperature
        y = erf[cause]["mean"].loc[tz].values    # Relative Risk        
        axs[i].plot(x, y, label=tz)
        
    axs[i].axhline(y=0.99, color="grey", linestyle="--", zorder=1)

    pltkit.StylizeAxes(
        axs[i],
        facecolor="whitesmoke",
        grid=True, 
        grid_kwargs={"c":"white"}, 
        title=f"{pltkit.causes[cause]}", 
        title_kwargs={"fontsize":10, "fontweight":"bold", "y":1.02}, 
        spines={"top":False, "right":False, "left":False, "bottom":False},
        ylim=(0.75,2)
        )
    
    for i in [0, 4, 8, 12, 16]:
        pltkit.StylizeAxes(
            axs[i],
            ylabel="Relative Risk [RR]",
            ylabel_kwargs={"fontsize":8}
            )
    for i in range(13,17):
        pltkit.StylizeAxes(
            axs[i],
            xlabel="Daily mean temperature [°C]",
            xlabel_kwargs={"fontsize":8}
            )
        
for i in [17, 18, 19]:
    axs[i].axis('off')

mpl.pyplot.tight_layout()
handles, labels = axs[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.65, 0.15), ncol=8, title="Temperature zones", title_fontsize=10)

mpl.pyplot.savefig(wdir +"/figures/Paper1/SM_Burkart_ERFs.pdf", dpi=300, bbox_inches='tight')

In [ ]:
"""
RESULTS TABLE APPENDIX
"""

filename = "mortality_ReplicationBurkart_SSP2_ERA5_1990-2019_*"
region_type="IMAGE"
region = "World"
t_type = "all"
cause = "All causes"
age_group = "All ages"
variable = "mortality"

bur_mean, bur_p025, bur_p975 = pltkit.LoadMortalityDraws(wdir+"burkart2022/output/ReplicationBurkart", filename, region_type, region, t_type, cause, age_group, variable)
print(f"Mean: 1990: {bur_mean.sel(year=1990).values}, 2000: {bur_mean.sel(year=2000).values}, 2010: {bur_mean.sel(year=2010).values}, 2019: {bur_mean.sel(year=2019).values}")
print(f"P2.5: 1990: {bur_p025.sel(year=1990).values}, 2000: {bur_p025.sel(year=2000).values}, 2010: {bur_p025.sel(year=2010).values}, 2019: {bur_p025.sel(year=2019).values}")
print(f"P97.5: 1990: {bur_p975.sel(year=1990).values}, 2000: {bur_p975.sel(year=2000).values}, 2010: {bur_p975.sel(year=2010).values}, 2019: {bur_p975.sel(year=2019).values}")


colors = ["#8C9A9E", "#6F1A07", "#5B9279", "#022B3A", "#E3B448"]
fig, ax = mpl.pyplot.subplots(figsize=(6,5))

mpl.pyplot.plot(bur_mean.year, bur_mean, label="Burkart et al., 2021", linewidth=2, c=colors[0])
mpl.pyplot.fill_between(bur_mean.year, bur_p025, bur_p975, color=colors[0], alpha=0.3)


mpl.pyplot.title(f"{region} {variable} estimations | {t_type.capitalize()} | {age_group.capitalize()} age group")
mpl.pyplot.legend(frameon=False)
mpl.pyplot.ylabel("Excess mortality (thousand people)")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

mpl.pyplot.tight_layout()
mpl.pyplot.show()

### Carleton 2022

In [ ]:
"""
IMPACT REGIONS
"""
ir = gpd.read_file(wdir + "models/carleton2022/data/CarletonSM/ir_shp/impact-region.shp")


fig = mpl.pyplot.figure(figsize=(10,5), dpi=300)
ax = fig.add_subplot(111, projection=cartopy.crs.Robinson(central_longitude=0), frameon=True)
ax.set_rasterization_zorder(3)

ax.coastlines(resolution='10m', lw = 0.1)
gl = ax.gridlines(crs=cartopy.crs.PlateCarree(), draw_labels=False, linewidth=0, color='white', alpha=0.5)
ax.spines['geo'].set_linewidth(0.2)

im = ir.boundary.plot(
    ax=ax,
    transform=cartopy.crs.PlateCarree(),
    color="black",
    linewidth=0.05
    )
ax.add_feature(cartopy.feature.OCEAN, facecolor='white', zorder=2)

mpl.pyplot.savefig(wdir + "figures/Paper1/SM_Carleton_ir.pdf", dpi=300, bbox_inches='tight')
mpl.pyplot.show()

In [ ]:
"""
Carleton ERFs
"""

# Load ERFs
flattened_config = {}
flattened_config["wdir"] = wdir+"models/carleton2022"
flattened_config["temp_dir"] = "X:/user/liprandicn/Data/ERA5/t2m_daily"
flattened_config["gdp_dir"] = None
flattened_config["project"] = "default"
flattened_config["scenario"] = "SSP2_L"
flattened_config["start_year"] = 1980
flattened_config["end_year"] = 2015
flattened_config["adaptation"] = None
flattened_config["counterfactual"] = None
flattened_config["reporting_tool"] = None
flattened_config["draw"] = "mean"
flattened_config["monthly_output"] = False
flattened_config["impact_regions"] = False
flattened_config["dask_on"] = False
flattened_config["stochastic"] = False

sets = mf.ModelSettings(**flattened_config)
base = mf.LoadInputData.for_baseline(sets=sets)
gamma_coeffs = mf.ImportGammaCoefficients(sets, sets.draw)

erfs_t0, tmin_t0 = mf.GenerateERFAll(
    sets=sets, 
    base=base,
    tempe=None,
    scen=None,
    erf=None,
    gammas=gamma_coeffs,
    year=None, 
    adaptation=False, 
    counterfactual=None
    ) 

fig, axs = mpl.pyplot.subplots(1,3, figsize=(12,4), dpi=300)
axs = axs.flatten()
t = np.arange(-20, 40.1, 0.1).round(1)
age_groups = ["+65 years", "5-64 years", "0-4 years"]

for i, age in enumerate(["oldest", "older", "young"]):
    
    axs[i].set_rasterization_zorder(3)
    axs[i].plot(t, erfs_t0[:,i].mean(axis=0), color="k", linewidth=2, zorder=3)
    for j in range(24378):
        axs[i].plot(t, erfs_t0[j,i], color="silver", alpha=0.1, linewidth=0.1, zorder=2)
    pltkit.StylizeAxes(
        axs[i], 
        facecolor="white",
        title=age_groups[i],
        title_kwargs={"fontsize": 12, "fontweight": "bold"},
        xlabel="Daily temperature (°C)",
        ylim=(0, 16),
        xlim=(-20, 40),
        spines={"top":False, "right":False}
    )
axs[0].set_ylabel("Relative mortality \n [deaths per 100,000 people]", fontsize=12)
mpl.pyplot.tight_layout()

mpl.pyplot.savefig(wdir+"/figures/Paper1/SM_Carleton_ERFs.pdf", dpi=300, bbox_inches='tight')
mpl.pyplot.show()

In [ ]:
"""
Replication Figure
"""

mor_2050 = pd.read_csv(wdir+'models/carleton2022/data/CarletonSM/erf_replication/TMEANt_logGDPt_oldest_2050.csv', index_col=[0])
mor_2100 = pd.read_csv(wdir+'models/carleton2022/data/CarletonSM/erf_replication/TMEANt_logGDPt_oldest_2100.csv', index_col=[0])
oldest = pd.read_csv(wdir+'models/carleton2022/data/CarletonSM/erf_replication/oldest.csv', index_col=[1])

delta_2050 = mor_2050['35.0'] - oldest['35.0']
delta_2100 = mor_2100['35.0'] - oldest['35.0']

ir = gpd.read_file(wdir+"models/carleton2022/data/CarletonSM/ir_shp/impact-region.shp")

ir['delta_2050'] = delta_2050.values
ir['delta_2100'] = delta_2100.values

ir = ir[~ir["hierid"].str.contains("ATA")]

fig, ax = mpl.pyplot.subplots(1, 2, figsize=(10, 4), dpi=300)

vmin, vmax = -21, 2
cmap = 'YlGn_r'

ir.plot(column='delta_2050', ax=ax[0], legend=False, cmap=cmap, vmin=vmin, vmax=vmax, rasterized=True)
ax[0].set_title('2050', fontsize=8)
ax[0].axis('off')

ir.plot(column='delta_2100', ax=ax[1], legend=False, cmap=cmap, vmin=vmin, vmax=vmax, rasterized=True)
ax[1].set_title('2100', fontsize=8)
ax[1].axis('off')

mpl.pyplot.subplots_adjust(wspace=0.01, bottom=0.25) 

cax = fig.add_axes([0.4, 0.3, 0.3, 0.03]) 
cbar = fig.colorbar(ax[1].collections[0], cax=cax, orientation='horizontal')
cbar.ax.tick_params(labelsize=7, size=3, width=0.5)

cbar.set_label('Change in relative mortality \n [deaths per 100,000 people]', fontsize=8)
mpl.pyplot.savefig(wdir+"/figures/Paper1/SM_Carleton_Replication.pdf", dpi=300, bbox_inches='tight')
mpl.pyplot.show()

### Honda 2014

In [ ]:
"""
Honda ERF
"""

df_interp = pd.read_csv(wdir + "models/Honda2014/data/risk_function/RiskFunction_Honda.csv")

fig, ax = mpl.pyplot.subplots(figsize=(6,5))
ax.plot(df_interp["daily_temperature"], df_interp["relative_risk_mean"], color="k")
ax.fill_between(
    df_interp["daily_temperature"], 
    df_interp["relative_risk_lower"], 
    df_interp["relative_risk_upper"], 
    alpha=0.2, color="k")
pltkit.StylizeAxes(
    ax=ax,
    spines={"top":False, "right":False},
    xlabel="Daily temperature (°C) - OT",
    ylabel="Relative Risk",
    title="Risk function of excess mortality due to heat",
    title_kwargs={"fontsize": 12, "y":1.05},
)

mpl.pyplot.tight_layout()
mpl.pyplot.savefig(wdir+"/figures/Paper1/SM_Honda_ERF.pdf", dpi=300, bbox_inches='tight')

In [ ]:
"""
Optimal temperatures map
"""

ot = xr.open_dataset(wdir+"models/honda2014/data/optimal_temperatures/era5_t2m_max_1980-2010_p84.nc")

fig = mpl.pyplot.figure(figsize=(10,5), dpi=300)
ax = fig.add_subplot(111, projection=cartopy.crs.Robinson(central_longitude=0), frameon=True)
ax.set_rasterization_zorder(3)
ax.coastlines(resolution='10m', lw = 0.1)
gl = ax.gridlines(crs=cartopy.crs.PlateCarree(), draw_labels=False, linewidth=0, color='white', alpha=0.5)
ax.spines['geo'].set_linewidth(0.2)

im = ot.t2m_p84.plot(ax=ax, 
                transform=cartopy.crs.PlateCarree(), 
                cmap='RdBu_r', 
                add_colorbar=False,
                levels=21, 
                vmin=-40, 
                vmax=40)
ax.add_feature(cartopy.feature.OCEAN, facecolor='white', zorder=2)

# Manually set colorbar limits
cbar = fig.colorbar(im, ax=ax, orientation='vertical', shrink=0.6, pad=0.05, aspect=20)
cbar.ax.set_ylim(-12, 36) 
cbar.ax.yaxis.set_major_formatter(mpl.ticker.FormatStrFormatter('%d°C'))

# Remove Antarctica 
land_shp = cartopy.io.shapereader.natural_earth(resolution='110m', category='physical', name='land')
land_geoms = cartopy.io.shapereader.Reader(land_shp).geometries()
for land in land_geoms:
    if land.bounds[1] < -60:
        ax.add_geometries([land], cartopy.crs.PlateCarree(), 
                          facecolor='whitesmoke', 
                          edgecolor='none', 
                          zorder=2)
        
mpl.pyplot.title('Optimal daily maximum temperatures (OT)', fontsize=10)
mpl.pyplot.savefig(wdir+"figures/Paper1/SM_Honda_Map.pdf", dpi=300, bbox_inches='tight')
mpl.pyplot.show()

In [ ]:
"""
Replication Romanello 2024
"""

filename = "models/honda2014/output/ReplicationRomanello/mortality_ReplicationRomanello_SSP2_ERA5_2000-2023"

rt = "IMAGE"
temp_type = "heat"
age_group = "oldest"
region = "World"
cause = "All causes"
var = "mortality"

rom = pltkit.LoadMortality(wdir, filename, rt, region, temp_type, cause, age_group, var)

fig = mpl.pyplot.figure(figsize=(10,5), dpi=300)
mpl.pyplot.plot(rom.year, rom.sel(var_mor="mean"), label="Romanello et al., 2024", linewidth=2, c="k")
mpl.pyplot.fill_between(rom.year, rom.sel(var_mor="lower"), rom.sel(var_mor="upper"), alpha=0.2, color="k")

formatter = mpl.ticker.ScalarFormatter(useMathText=True)
formatter.set_scientific(True)
formatter.set_powerlimits((5, 5))
mpl.pyplot.gca().yaxis.set_major_formatter(formatter)
mpl.pyplot.grid(False)
mpl.pyplot.ylim(0, 5e5)
mpl.pyplot.title("Heat-related mortality")
mpl.pyplot.ylabel("Excess mortality over 65")

mpl.pyplot.savefig(wdir + "figures/Paper1/SM_Honda_Replication.pdf", dpi=300, bbox_inches='tight')
mpl.pyplot.show()

### Scovronick 2024

In [ ]:
"""
Scoronick ERFs
"""

result = pyreadr.read_r(wdir + "models/scovronick2024/data/Scovronick_SM/Fig2_20Nov2025.RData")

fig, axs = mpl.pyplot.subplots(2, 2, figsize=(9,7), dpi=300)
axs = axs.flatten()

groups = ['40', '55', '70', '85']
diseases = ['ncrc', 'rsp', 'cvd', 'all']
cause_name = ['Non-cardiorespiratory diseases', 'Respiratory diseases', 'Cardiovascular diseases', 'All-cause mortality']

colors = mpl.pyplot.cm.tab10(range(len(groups))) 

for i, disease in enumerate(diseases):
    for j, group in enumerate(groups):
        color = colors[j]

        x = result[disease]['Percentile']
        y = result[disease][f'{disease}_{group}_rr']
        low = result[disease][f'{disease}_{group}_rr_low']
        high = result[disease][f'{disease}_{group}_rr_high']

        # Central line
        axs[i].plot(x, y, label=f'Age {group}', color=color, zorder=2)
        # Uncertainty range
        axs[i].fill_between(x, low, high, color=color, alpha=0.25, zorder=2)

    pltkit.StylizeAxes(
        axs[i], 
        title=cause_name[i],
        ylim=(0.9, 1.65),
        spines={'top': False, 'right': False},
        grid=True, 
        grid_kwargs={"c":"white"},
        )

    if i == 0 or i == 2:
        axs[i].set_ylabel('Relative Risk (RR)', fontsize=10)
    if i == 2 or i == 3:
        axs[i].set_xlabel('Percentile of Temperature', fontsize=10)

    axs[i].axhline(1.0, linestyle='--', color='silver')

handles, labels = axs[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.5, 0.0), ncol=8, title="Age", title_fontsize=10)

mpl.pyplot.tight_layout()
mpl.pyplot.savefig(wdir + "/figures/Paper1/Scovronick_ERFs.pdf", dpi=300, bbox_inches='tight')
mpl.pyplot.show()

### Comparison

In [53]:
"""
Params
"""

temp_type = "heat"
age_group = "oldest"
cause = "All causes"
rt = "IMAGE"
var = "mortality"

models = {
    "hon_file" : [
        0,
        "honda2014/output/ComparisonHonda/mortality_ComparisonHonda_SSP2_ERA5_1980-2023_counterfactual",
        "Honda et al., 2014",
        "H"
        ],
    "sco_file" : [
        1,
        "scovronick2024/output/ComparisonScovronick/mortality_ComparisonScovronick_SSP2_ERA5_1980-2023_counterfactual",
        "Scovronick et al., 2024",
        "S"
        ],
    "car_file" : [
        2,
        "carleton2022/output/ComparisonCarletonCounter/mortality_ComparisonCarletonCounter_SSP2_ERA5_NoAdap_1980-2023_*",
        "Carleton et al., 2022",
        "C"
        ],       
    "bur_file" : [
        3,
        "burkart2022/output/ComparisonBurkart/mortality_ComparisonBurkart_SSP2_ERA5_1980-2023_counterfactual_*",
        "Burkart et al., 2021",
        "B"
        ]
    }

In [4]:
# Load regional values for insets
# Run only once

# region_vals={}
# for region in mpl.pyplotkit.IMAGE_REGIONS:
#     region_vals[region] = {"main": {}, "lower": {}, "upper": {}}
#     for i,model in enumerate(models):
#         m, l, u = mpl.pyplotkit.LoadMortalityDraws(wdir+"models/", models[model][1], rt, region, temp_type, cause, age_group, var, None)
#         region_vals[region]["main"][model] = m.values
#         region_vals[region]["lower"][model] = l.values
#         region_vals[region]["upper"][model] = u.values
        
# years=range(1980,2024)

# rows = []
# for region, sub_dict in region_vals.items():
#     for main_key, files_dict in sub_dict.items():
#         for file_name, array_data in files_dict.items():
#             for val, year in zip(array_data, years):
#                 rows.append({
#                     'Region': region,
#                     'Type': main_key,
#                     'File': file_name,
#                     'Value': val,
#                     "Year":year
#                 })

# df = pd.DataFrame(rows)

# df = df.pivot_table(
#     index=['Region', 'File', "Type"], 
#     columns='Year', 
#     values='Value'
# ).reset_index()

# df.to_csv(wdir+"figures/Paper1/Appendix_HistoricalMortality_Continents.csv", index=False)

In [ ]:
df = pd.read_csv(wdir+"figures/Paper1/Appendix_HistoricalMortality_Continents.csv")

continents= {
    "Northern America": ["CAN", "USA"],
    "Europe": ["WEU", "CEU", "UKR"],
    "Asia": ["TUR", "STAN", "RUS", "ME", "INDIA", "KOR", "CHN", "SEAS", "INDO", "JAP", "RSAS"],
    "Latin America and the Caribbean": ["MEX", "RCAM", "BRA", "RSAM"],
    "Africa": ["NAF", "WAF", "EAF", "SAF", "RSAF"],
    "Oceania": ["OCE"]
    }

region_to_continent = {
    region: continent 
    for continent, regions in continents.items() 
    for region in regions
}
df['Continent'] = df['Region'].map(region_to_continent)
df = df.groupby(['Continent', 'File', 'Type']).sum(numeric_only=True).reset_index()
continents.keys()

fig, axs = mpl.pyplot.subplots(2,3, figsize=(10,6), dpi=300)
axs = axs.flatten()

colors_list = ["#C8553D", "#566E3D", "#222E50", "#FEA82F", "#829191"]

labelsize=10
titlesize=11
lettersize=7
years_vlines = [1998, 2003, 2010, 2016, 2019, 2022]

for i,region in enumerate(list(continents.keys())):
    for j, model in enumerate(models):
        main = df[(df['Continent'] == region) & (df['File'] == model) & (df["Type"]=="main")].iloc[:,3:]
        lower = df[(df['Continent'] == region) & (df['File'] == model) & (df["Type"]=="lower")].iloc[:,3:]
        upper = df[(df['Continent'] == region) & (df['File'] == model) & (df["Type"]=="upper")].iloc[:,3:]
        
        axs[i].plot(main.columns.astype(int), main.values[0], label=models[model][2], linewidth=1, c=colors_list[j], clip_on=False)
        fill = axs[i].fill_between(main.columns.astype(int), lower.values[0], upper.values[0], color=colors_list[j], alpha=0.3)
        fill.set_clip_on(False)

    for year in years_vlines:
        axs[i].axvline(x=year, color='gray', linestyle='--', linewidth=1, alpha=0.7)
        
    axs[i].set_title(f"{region}", fontsize=titlesize, y=1.1)
    axs[i].tick_params(axis='both', labelsize=labelsize)
    axs[i].yaxis.set_major_formatter(mpl.ticker.FuncFormatter(lambda x, pos: f'{x/1000:g}k'))
    axs[i].spines["top"].set_visible(False)
    axs[i].spines["right"].set_visible(False)
    
    if (i==0) or (i==3):
        axs[i].set_ylabel("Global excess mortality", fontsize=labelsize)
        
axs[0].set_ylim(-10e3,25e3)
axs[1].set_ylim(-2e3,100e3)
axs[4].set_ylim(-10e3,105e3)
axs[3].set_ylim(-10e3,96e3)
axs[2].set_ylim(-100e3,500e3)
axs[5].set_ylim(-0.5e3,2.1e3)


handles, labels = axs[0].get_legend_handles_labels()
fig.legend(
    handles=handles,
    labels=labels,
    loc='lower center',
    bbox_to_anchor=(0.5, 0.0),
    ncol=len(models),
    frameon=False,
    fontsize=labelsize
)

mpl.pyplot.tight_layout(rect=[0, 0.06, 1, 1])
mpl.pyplot.savefig(wdir+"figures/Paper1/SM_Comparison_Continents.pdf", dpi=300, bbox_inches='tight')
mpl.pyplot.show()

### Future projections

In [ ]:
"""
ScenarioMIP7 temperature projections
"""

smip_temperature  = {}
smip_temperature_magicc = {}
smip_path = "X:/user/doelmanj/ScenDevelopment/ScenarioMIP/3_IMAGE_land/scen/"

timeline = prism.Timeline(
                start=prism.Q_(1970,"year"), # Can be changed
                end=prism.Q_(2100,"year"),
                stepsize=prism.Q_(1, "year")
                )
SCENARIOS = [
    "SSP1_M_CP", "SSP1_M", "SSP1_ML", "SSP1_L", "SSP1_VLLO", "SSP1_VLHO", 
    "SSP2_M_CP", "SSP2_M", "SSP2_ML", "SSP2_L", "SSP2_VLLO", "SSP2_VLHO",
    "SSP3_M_CP", "SSP3_H",
    "SSP5_H", "SSP5_HL"
    ]

for scenario in SCENARIOS:
    
    scenariomip_temperature = smip_path + scenario + "/output/climate_impacts/TEMPERATURE.OUT"
    
    smip_temperature[scenario] = prism.TimeVariable(
        dims = (),
        unit = "K",
        file = scenariomip_temperature,
        timeline = timeline
    )

ssp_styles = {
    "SSP1": "-", "SSP2": "--", "SSP3": "-.", "SSP5": ":"
}

target_colors = {
    "VLLO": "#4ea8de", 
    "VLHO": "#0077b6",  

    "L":    "#52b788",  
    "ML":   "#95d5b2",  
    
    "M":    "#f4a261", 
    "M_CP": "#e9c46a",  
    
    "H":    "#e76f51", 
    "HL":   "#ff85a1"  
}

fig, ax = mpl.pyplot.subplots(figsize=(10,6))
texts = []

for scenario in SCENARIOS:
    
    parts = scenario.split("_", 1)
    ssp = parts[0]
    target = parts[1]
    
    line_style = ssp_styles.get(ssp, "-")
    line_color = target_colors.get(target, "black")
    
    data_slice = smip_temperature[scenario].to_array().sel(time=slice(2020, 2100))
    data_slice.plot(ax=ax, color=line_color, linestyle=line_style, alpha=0.7)
    
    x_end = 2100
    y_end = data_slice.sel(time=x_end).values.item()
    
    t = ax.text(x_end, y_end, scenario, color=line_color, va="center", fontsize=9, fontweight='bold')
    texts.append(t)

ax.set_xlim(2025, 2109) 
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.set_ylabel("ΔGMST compared to 1850-1900 [K]", fontsize=10)
ax.set_xlabel("")

adjust_text(
    texts, 
    only_move={'text': 'y'},     
    force_points=0.0,            
    expand=(1.0, 1.1),            
)

mpl.pyplot.title("ScenarioMIP7 scenarios", fontsize=11)
mpl.pyplot.savefig(wdir+"figures/Paper1/SM_ScenarioMIP7.pdf", dpi=300, bbox_inches='tight')
mpl.pyplot.show()

In [ ]:
model = "ACCESS-CM2"
var_dir = "X:/user/scherrenbm/Data/Internal_Variability_EERIE/"
file_list = sorted(glob.glob(var_dir+f"//internal_variability_grid_tas_Amon_{model}_hist*.nc"))


fig, ax = mpl.pyplot.subplots(3,3, figsize=(12,15))
ax=ax.flatten()

for i in range(len(file_list)):
    filename = re.search(rf'{model}_hist-(.*?)_gn', file_list[i]).group(1)
    var = xr.open_dataset(file_list[i], decode_times=False).mean(dim="lat").mean(dim="lon").isel(time=slice(-1212, None))
    ax[i].plot(pd.date_range(start='2000-01-01', end='2100-12-31', freq='ME'), var.IV_tas.values, label=filename, c=f"grey", linewidth=0.5)
    ax[i].legend(loc="upper left")
    ax[i].set_ylim(-2,3)
    if i in [0,3,6]:
        ax[i].set_ylabel("ΔT [°C]")
        
mpl.pyplot.suptitle(f"{model}  internal variability", y=0.91, fontsize=12)
mpl.pyplot.savefig(wdir+"figures/Paper1/SM_Climate_Variability.pdf", dpi=300, bbox_inches='tight')
mpl.pyplot.show()

### Shapley-Owen